In [4]:
import pandas as pd
import numpy as np

In [6]:
import pandas as pd

df = pd.read_csv("C:/Users/SwapnilShinde/OneDrive/Documents/Study Material/Sem 6/AML/Experiment Codes/Ex09/Data/NewsDataset.csv")

print(df.head())

                                             Summary  \
0  Celeb chef Anthony Bourdain wins multiple Emmy...   
1  Clearly something is wrong with US, says JPMor...   
2  Two more allegedly commit suicide over Maratha...   
3  Must curb hate speeches during poll campaigns:...   
4  PNB to block all Maestro debit cards from July 31   

                                                Text  
0  Celebrity chef Anthony Bourdain, who passed aw...  
1  World's most valuable bank JPMorgan Chase's CE...  
2  Two more people have allegedly committed suici...  
3  A group of former chief election commissioners...  
4  Punjab National Bank (PNB) Maestro debit card ...  


In [8]:
import re
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z ]", "", text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

df['Text'] = df['Text'].apply(clean_text)
df['Summary'] = df['Summary'].apply(clean_text)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\SwapnilShinde\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [9]:
df = df.sample(8000, random_state=42)
df.reset_index(drop=True, inplace=True)

In [10]:
max_text_len = 80
max_summary_len = 15

In [12]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Tokenizers
text_tokenizer = Tokenizer(num_words=50000)
summary_tokenizer = Tokenizer(num_words=20000)

text_tokenizer.fit_on_texts(df['Text'])
summary_tokenizer.fit_on_texts(df['Summary'])

# Convert to sequences
text_seq = text_tokenizer.texts_to_sequences(df['Text'])
summary_seq = summary_tokenizer.texts_to_sequences(df['Summary'])

# Padding
X = pad_sequences(text_seq, maxlen=max_text_len, padding='post')
y = pad_sequences(summary_seq, maxlen=max_summary_len, padding='post')

In [13]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.1, random_state=42
)

In [14]:
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense
from tensorflow.keras.models import Model

latent_dim = 128  # reduced for speed

# Encoder
encoder_inputs = Input(shape=(max_text_len,))
enc_emb = Embedding(50000, latent_dim)(encoder_inputs)
encoder_lstm = LSTM(latent_dim, return_state=True)
_, state_h, state_c = encoder_lstm(enc_emb)

# Decoder
decoder_inputs = Input(shape=(max_summary_len-1,))
dec_emb = Embedding(20000, latent_dim)(decoder_inputs)
decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=[state_h, state_c])

decoder_dense = Dense(20000, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

In [15]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [17]:
# Run this FIRST

y_train_in = y_train[:, :-1]
y_train_out = y_train[:, 1:]

y_val_in = y_val[:, :-1]
y_val_out = y_val[:, 1:]

In [18]:
import numpy as np

y_train_out = np.reshape(y_train_out, (y_train_out.shape[0], y_train_out.shape[1], 1))
y_val_out = np.reshape(y_val_out, (y_val_out.shape[0], y_val_out.shape[1], 1))

In [19]:
history = model.fit(
    [X_train, y_train_in],
    y_train_out,
    epochs=8,
    batch_size=64,
    validation_data=([X_val, y_val_in], y_val_out)
)

Epoch 1/8
113/113 ━━━━━━━━━━━━━━━━━━━━ 38s 315ms/step - accuracy: 0.5487 - loss: 5.4448 - val_accuracy: 0.5539 - val_loss: 4.1292
Epoch 2/8
113/113 ━━━━━━━━━━━━━━━━━━━━ 36s 314ms/step - accuracy: 0.5538 - loss: 3.9636 - val_accuracy: 0.5539 - val_loss: 4.1077
Epoch 3/8
113/113 ━━━━━━━━━━━━━━━━━━━━ 36s 318ms/step - accuracy: 0.5539 - loss: 3.8896 - val_accuracy: 0.5542 - val_loss: 4.1115
Epoch 4/8
113/113 ━━━━━━━━━━━━━━━━━━━━ 35s 311ms/step - accuracy: 0.5544 - loss: 3.8426 - val_accuracy: 0.5546 - val_loss: 4.1291
Epoch 5/8
113/113 ━━━━━━━━━━━━━━━━━━━━ 35s 310ms/step - accuracy: 0.5548 - loss: 3.8050 - val_accuracy: 0.5551 - val_loss: 4.1439
Epoch 6/8
113/113 ━━━━━━━━━━━━━━━━━━━━ 35s 310ms/step - accuracy: 0.5550 - loss: 3.7675 - val_accuracy: 0.5545 - val_loss: 4.1609
Epoch 7/8
113/113 ━━━━━━━━━━━━━━━━━━━━ 35s 312ms/step - accuracy: 0.5553 - loss: 3.7279 - val_accuracy: 0.5548 - val_loss: 4.1942
Epoch 8/8
113/113 ━━━━━━━━━━━━━━━━━━━━ 35s 312ms/step - accuracy: 0.5556 - loss: 3.6889 - 

In [20]:
reverse_target_word_index = summary_tokenizer.index_word

target_word_index = summary_tokenizer.word_index

In [26]:
encoder_model = Model(encoder_inputs, [state_h, state_c])

In [27]:
# Decoder states input
decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))

decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

# Decoder embedding
dec_emb2 = Embedding(20000, latent_dim)(decoder_inputs)

# Decoder LSTM
decoder_outputs2, state_h2, state_c2 = decoder_lstm(
    dec_emb2, initial_state=decoder_states_inputs
)

decoder_states = [state_h2, state_c2]

# Dense
decoder_outputs2 = decoder_dense(decoder_outputs2)

# Final decoder model
decoder_model = Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs2] + decoder_states
)

In [28]:
def generate_summary(input_text):
    input_seq = text_tokenizer.texts_to_sequences([input_text])
    input_seq = pad_sequences(input_seq, maxlen=max_text_len, padding='post')

    # Use encoder MODEL (not layer)
    states_value = encoder_model.predict(input_seq)

    target_seq = np.zeros((1,1))
    target_seq[0,0] = target_word_index['startseq']

    summary = ""

    while True:
        output_tokens, h, c = decoder_model.predict(
            [target_seq] + states_value
        )

        sampled_index = np.argmax(output_tokens[0, -1, :])
        sampled_word = reverse_target_word_index.get(sampled_index, "")

        if sampled_word == "endseq" or len(summary.split()) > max_summary_len:
            break

        summary += " " + sampled_word

        target_seq = np.zeros((1,1))
        target_seq[0,0] = sampled_index

        states_value = [h, c]

    return summary

In [31]:
print("Original:\n", df['Text'][0])
print("\nGenerated Summary:\n", generate_summary(df['Text'][0]))

Original:
 talinda bennington wife rock band linkin parks singer chester bennington took twitter share video singer hours suicide talinda wrote depression doesnt face mood depression looked like us hrs death video chester seen playing game family
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step


KeyError: 'startseq'